This notebook implements the naive Mono-Forward Algorithm with the gradients

In [6]:
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as T
from typing import List, Optional, Tuple


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE

device(type='cuda')

In [ ]:
class MonoFwdLinearBlock(nn.Module):
    def __init__(self,in_dim: int, out_dim : int, num_classes: int, activation:str = "relu"):
        super().__init__()    
        self.linear = nn.Linear(in_dim, out_dim)

        # the projection matrix, where m = num categories, n = number of neurons
        m = num_classes
        n = out_dim
        self.M = nn.Parameter(torch.randn(m,n))

        self.activation =  F.relu if activation == "relu" else F.tanh
        

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor,torch.Tensor]:
        a = self.linear(x)
        g = a @ self.M.T
        return self.activation(a),g

class MonoFwdMLP(nn.Module):

    def __init__(self,input_dim : int, hidden_dims: list[int], num_classes: int, activation : str = "relu"):
        super.__init__()    
        dims = [input_dim] + hidden_dims
        self.blocks = nn.ModuleList(
            [MonoFwdLinearBlock(dims[i], dims[i + 1], num_classes, activation=activation) for i in range(len(hidden_dims))]
        )
        self.num_classes = num_classes
    
    def local_losses(self, x:torch.Tensor, y:torch.Tensor):
        
        if x.dim() > 2:
            x = x.flatten(1)
        
        losses: List[torch.Tensor] = []
        goodness_per_layer: List[torch.Tensor] = []
        
        h = x
        for block in self.blocks:
            a,g = block.forward(h)
            # paper ref -> cross entropy LF
            losses.append(F.cross_entropy(g,y))
            goodness_per_layer.append(g)
            h = a.detach()
        
        return losses,goodness_per_layer
